In [1]:
import sys
!{sys.executable} -m pip install -q pandas==2.2.3 requests==2.32.3 beautifulsoup4==4.12.3 lxml==5.3.0 rapidfuzz==3.9.7 tqdm==4.66.5


In [2]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import re
import time
import unicodedata
import xml.etree.ElementTree as ET

from bs4 import BeautifulSoup
import pandas as pd
import requests
from rapidfuzz import fuzz, process
from tqdm.auto import tqdm


C:\Users\Giovanny\anaconda3\envs\fluidgpu\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
CWD = Path.cwd().resolve()
BASE_DIR = CWD if CWD.name == "notebooks" else CWD / "notebooks"
OUTPUT_DIR = BASE_DIR / "output"
CACHE_DIR = BASE_DIR / "cache"
RAW_DIR = BASE_DIR / "raw"
REPORTS_DIR = BASE_DIR / "reports"
OFAC_RAW_DIR = RAW_DIR / "ofac"
UN_RAW_DIR = RAW_DIR / "un"

for directory in [BASE_DIR, OUTPUT_DIR, CACHE_DIR, RAW_DIR, REPORTS_DIR, OFAC_RAW_DIR, UN_RAW_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

FORCE_DOWNLOAD = False
REQUEST_TIMEOUT = 60
HEADERS = {"User-Agent": "pep-public-intel-notebook/1.0 public-osint-research contact: local-notebook"}
print(f"BASE_DIR: {BASE_DIR}")


BASE_DIR: C:\Users\Giovanny\Desktop\web_scraping_peps\notebooks


In [4]:
INPUT_CONTEXT_COLUMNS = [
    "estado_input", "municipio_input", "puesto_input", "dependencia_input", "periodo_inicio_input",
    "periodo_fin_input", "partido_input", "rfc_input", "curp_input", "empresa_relacionada_input", "alias_input",
]

EVIDENCE_COLUMNS = [
    "seed_id", "nombre_persona_input", "normalized_name_input", "estado_input", "municipio_input",
    "puesto_input", "dependencia_input", "periodo_inicio_input", "periodo_fin_input", "partido_input",
    "rfc_input", "curp_input", "empresa_relacionada_input", "alias_input",
    "source", "source_type", "source_country", "source_official", "source_category", "source_reliability",
    "matched_record_name", "matched_record_normalized_name", "matched_alias", "matched_entity_type",
    "matched_role", "matched_position", "matched_agency", "matched_state", "matched_municipality",
    "matched_country", "matched_identifier", "matched_company", "matched_rfc", "matched_curp",
    "match_score_pct", "match_strength", "match_method", "matched_fields", "conflicting_fields",
    "requires_review", "identity_confidence_pct", "signal_type", "signal_category", "severity",
    "risk_layer", "risk_level_candidate", "confidence_pct", "evidence_title", "evidence_summary",
    "evidence_snippet", "evidence_keywords", "evidence_date", "evidence_url", "source_url",
    "search_query", "search_engine", "retrieved_at", "raw_file_path", "raw_file_sha256",
    "involvement_summary", "explanation", "limitations", "recommended_action",
]

SUMMARY_COLUMNS = [
    "seed_id", "nombre_persona_input", "total_sanctions_candidates", "ofac_candidates", "un_candidates",
    "max_match_score_pct", "max_identity_confidence_pct", "max_severity", "requires_review",
    "top_evidence_url", "top_involvement_summary", "recommended_action",
]

PARTICLES = {"de", "del", "la", "las", "los", "el", "y"}
SEVERITY_RANK = {"critical_candidate": 0, "high_review": 1, "medium_review": 2, "low_signal": 3, "insufficient_evidence": 4, "discard": 5}


def now_utc():
    return datetime.now(timezone.utc).isoformat()


def remove_accents(value):
    if pd.isna(value):
        return ""
    return "".join(ch for ch in unicodedata.normalize("NFKD", str(value)) if not unicodedata.combining(ch))


def normalize_text(value):
    text = remove_accents(value).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def normalize_name(value, remove_particles=False):
    tokens = normalize_text(value).split()
    if remove_particles:
        tokens = [token for token in tokens if token not in PARTICLES]
    return " ".join(tokens)


def stable_hash(value):
    return hashlib.sha256(normalize_name(value).encode("utf-8")).hexdigest()[:16]


def score_name_match(left, right):
    left_norm = normalize_name(left)
    right_norm = normalize_name(right)
    if not left_norm or not right_norm:
        return 0.0
    left_tokens = [token for token in left_norm.split() if token not in PARTICLES]
    right_tokens = [token for token in right_norm.split() if token not in PARTICLES]
    if not left_tokens or not right_tokens:
        return 0.0
    left_set = set(left_tokens)
    right_set = set(right_tokens)
    overlap = len(left_set & right_set)
    if overlap == 0:
        return 0.0
    smaller = min(len(left_set), len(right_set))
    larger = max(len(left_set), len(right_set))
    overlap_ratio = overlap / smaller if smaller else 0.0
    scores = [fuzz.WRatio(left_norm, right_norm), fuzz.token_sort_ratio(left_norm, right_norm), fuzz.ratio(left_norm, right_norm)]
    if smaller >= 2 and overlap_ratio >= 0.67:
        scores.append(fuzz.token_set_ratio(left_norm, right_norm))
    raw_score = float(max(scores))
    if smaller == 1 and larger >= 3:
        raw_score = min(raw_score, 59.0)
    elif smaller == 2 and larger >= 4:
        raw_score = min(raw_score, 74.0)
    if overlap_ratio < 0.50:
        raw_score = min(raw_score, 59.0)
    elif overlap_ratio < 0.67:
        raw_score = min(raw_score, 74.0)
    return round(raw_score, 2)


def classify_match(score):
    score = float(score or 0)
    if score >= 95:
        return "exact_or_very_strong_name"
    if score >= 88:
        return "strong_candidate"
    if score >= 75:
        return "medium_candidate"
    if score >= 60:
        return "weak_candidate"
    return "discard"


def match_method_for(seed_name, candidate_name, alias=False):
    if alias:
        return "alias_match"
    if normalize_name(seed_name) == normalize_name(candidate_name):
        return "exact_normalized_name"
    token_score = fuzz.token_sort_ratio(normalize_name(seed_name), normalize_name(candidate_name))
    partial_score = fuzz.partial_ratio(normalize_name(seed_name), normalize_name(candidate_name))
    return "token_sort_ratio" if token_score >= partial_score else "partial_ratio"


def empty_evidence_df():
    return pd.DataFrame(columns=EVIDENCE_COLUMNS)


def empty_summary_df():
    return pd.DataFrame(columns=SUMMARY_COLUMNS)


def input_has_only_name(seed):
    return bool(seed.get("has_only_name", False)) or not any(str(seed.get(col, "")).strip() for col in INPUT_CONTEXT_COLUMNS)


def context_assessment(seed, record):
    matched = ["nombre"]
    conflicts = []
    for input_col, record_col, label in [
        ("estado_input", "country", "estado/pais"),
        ("dependencia_input", "program_or_list", "dependencia/lista"),
        ("alias_input", "candidate_name", "alias"),
    ]:
        input_value = normalize_text(seed.get(input_col, ""))
        record_value = normalize_text(record.get(record_col, ""))
        if input_value and record_value and input_value in record_value:
            matched.append(label)
        elif input_value and record_value and label == "estado/pais" and input_value not in record_value:
            conflicts.append(f"{input_col}={seed.get(input_col, '')} no coincide con matched_country={record.get(record_col, '')}")
    return "; ".join(matched), "; ".join(conflicts)


def identity_confidence(score, seed, matched_fields, sensitive=True):
    confidence = float(score or 0) * 0.70
    extra_fields = [field.strip() for field in str(matched_fields).split(";") if field.strip() and field.strip() != "nombre"]
    confidence += min(len(extra_fields), 4) * 6
    if sensitive:
        confidence -= 10
    if input_has_only_name(seed):
        confidence = min(confidence, 75)
    if float(score or 0) < 95:
        confidence = min(confidence, 84)
    return round(max(0, min(confidence, 100)), 2)


def risk_and_action(score, confidence, source_type):
    if score >= 95 and confidence >= 85:
        return "high_review", "enhanced_due_diligence_review"
    if score >= 88:
        return "medium_review", "verify_identity"
    if score >= 75:
        return "low_signal", "manual_review"
    return "low_signal", "discard_likely_false_positive"


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


In [5]:
def fallback_seed_dataframe():
    print("ADVERTENCIA: no existe output/00_peps_normalized.csv; se crean datos minimos de prueba para esta corrida.")
    names = ["Claudia Sheinbaum Pardo", "Andres Manuel Lopez Obrador", "Marcelo Ebrard Casaubon", "Adan Augusto Lopez Hernandez", "Ricardo Monreal Avila", "Xochitl Galvez Ruiz", "Samuel Garcia Sepulveda", "Clara Brugada Molina", "Omar Garcia Harfuch", "Luisa Maria Alcalde Lujan"]
    rows = []
    for name in names:
        rows.append({
            "seed_id": stable_hash(name), "nombre_persona_input": name, "normalized_name_input": normalize_name(name),
            "estado_input": "", "municipio_input": "", "puesto_input": "", "dependencia_input": "",
            "periodo_inicio_input": "", "periodo_fin_input": "", "partido_input": "", "rfc_input": "", "curp_input": "",
            "empresa_relacionada_input": "", "alias_input": "", "has_only_name": True,
        })
    return pd.DataFrame(rows)

input_file = OUTPUT_DIR / "00_peps_normalized.csv"
seeds_df = pd.read_csv(input_file, encoding="utf-8-sig").fillna("") if input_file.exists() else fallback_seed_dataframe()
for col in ["seed_id", "nombre_persona_input", "normalized_name_input"] + INPUT_CONTEXT_COLUMNS + ["has_only_name"]:
    if col not in seeds_df.columns:
        seeds_df[col] = ""
print(f"Personas a consultar: {len(seeds_df):,}")
display(seeds_df.head(20))


Personas a consultar: 10


,seed_id,nombre_persona_input,normalized_name_input,token_sort_name,name_tokens,estado_input,municipio_input,puesto_input,dependencia_input,periodo_inicio_input,periodo_fin_input,partido_input,rfc_input,curp_input,empresa_relacionada_input,alias_input,has_only_name,input_quality_score,created_at
0,064afd0cc88b3699,Claudia Sheinbaum Pardo,claudia sheinbaum pardo,claudia pardo sheinbaum,claudia|sheinbaum|pardo,Nacional,,Presidenta de Mexico,Gobierno de Mexico,2024.0,,,,,,,False,0.77,2026-05-07T07:51:29.008342+00:00
1,910a01d167e16711,Andres Manuel Lopez Obrador,andres manuel lopez obrador,andres lopez manuel obrador,andres|manuel|lopez|obrador,Nacional,,Ex presidente de Mexico,Gobierno de Mexico,2018.0,2024.0,,,,,AMLO,False,0.99,2026-05-07T07:51:29.008342+00:00
2,55c8819cc4622591,Marcelo Ebrard Casaubon,marcelo ebrard casaubon,casaubon ebrard marcelo,marcelo|ebrard|casaubon,Nacional,,Servidor publico / figura publica,Gobierno de Mexico,,,,,,,,False,0.70,2026-05-07T07:51:29.008342+00:00
3,15b1d586cc02eb0e,Adan Augusto Lopez Hernandez,adan augusto lopez hernandez,adan augusto hernandez lopez,adan|augusto|lopez|hernandez,Tabasco,,Senador / figura publica,Senado de la Republica,,,,,,,,False,0.78,2026-05-07T07:51:29.008342+00:00
4,2597a81eb6062feb,Ricardo Monreal Avila,ricardo monreal avila,avila monreal ricardo,ricardo|monreal|avila,Zacatecas,,Legislador / figura publica,Congreso de la Union,,,,,,,,False,0.70,2026-05-07T07:51:29.008342+00:00
5,39162acdaa914138,Xochitl Galvez Ruiz,xochitl galvez ruiz,galvez ruiz xochitl,xochitl|galvez|ruiz,Nacional,,Senadora / figura publica,Senado de la Republica,,,,,,,,False,0.70,2026-05-07T07:51:29.008342+00:00
6,0cafd5ecac30f555,Samuel Garcia Sepulveda,samuel garcia sepulveda,garcia samuel sepulveda,samuel|garcia|sepulveda,Nuevo Leon,,Gobernador,Gobierno de Nuevo Leon,,,,,,,,False,0.70,2026-05-07T07:51:29.008342+00:00
7,cba9b1a8ead15ac3,Clara Brugada Molina,clara brugada molina,brugada clara molina,clara|brugada|molina,Ciudad de Mexico,,Jefa de Gobierno,Gobierno de la Ciudad de Mexico,,,,,,,,False,0.70,2026-05-07T07:51:29.008342+00:00
8,69573f0325299b8b,Omar Garcia Harfuch,omar garcia harfuch,garcia harfuch omar,omar|garcia|harfuch,Nacional,,Servidor publico / figura publica,Gobierno de Mexico,,,,,,,,False,0.70,2026-05-07T07:51:29.008342+00:00
9,75c154f2e7b3a23f,Luisa Maria Alcalde Lujan,luisa maria alcalde lujan,alcalde luisa lujan maria,luisa|maria|alcalde|lujan,Nacional,,Figura publica,Gobierno de Mexico,,,,,,,,False,0.78,2026-05-07T07:51:29.008342+00:00


In [6]:
def fetch_url_once(url, destination, force=False):
    if destination.exists() and not force:
        return {"ok": True, "path": destination, "url": url, "status": "cached", "seconds": 0.0, "sha256": sha256_file(destination), "error": "", "note": "raw reutilizado"}
    start = time.perf_counter()
    try:
        response = requests.get(url, headers=HEADERS, timeout=REQUEST_TIMEOUT)
        elapsed = time.perf_counter() - start
        if response.status_code in {401, 403}:
            return {"ok": False, "path": destination, "url": url, "status": str(response.status_code), "seconds": elapsed, "sha256": "", "error": "401/403", "note": "no se reintenta"}
        if response.status_code >= 400:
            return {"ok": False, "path": destination, "url": url, "status": str(response.status_code), "seconds": elapsed, "sha256": "", "error": f"HTTP {response.status_code}", "note": "no se reintenta"}
        content = response.content
        if b"captcha" in content[:5000].lower():
            return {"ok": False, "path": destination, "url": url, "status": "captcha", "seconds": elapsed, "sha256": "", "error": "captcha detectado", "note": "no se reintenta"}
        destination.write_bytes(content)
        return {"ok": True, "path": destination, "url": url, "status": "ok", "seconds": elapsed, "sha256": sha256_file(destination), "error": "", "note": "descargado"}
    except Exception as exc:
        elapsed = time.perf_counter() - start
        return {"ok": False, "path": destination, "url": url, "status": "error", "seconds": elapsed, "sha256": "", "error": str(exc)[:300], "note": "excepcion"}


def local_name(tag):
    return tag.split("}", 1)[-1]


def direct_text(element, child_name):
    for child in list(element):
        if local_name(child.tag).lower() == child_name.lower():
            return (child.text or "").strip()
    return ""


def all_desc_text(element, child_name):
    values = []
    for child in element.iter():
        if local_name(child.tag).lower() == child_name.lower() and child.text:
            value = child.text.strip()
            if value:
                values.append(value)
    return values


In [7]:
def parse_ofac_xml(path, source_label, source_url, raw_sha256):
    start = time.perf_counter()
    records = []
    root = ET.parse(path).getroot()
    for entry in root.iter():
        if local_name(entry.tag) != "sdnEntry":
            continue
        uid = direct_text(entry, "uid")
        first = direct_text(entry, "firstName")
        last = direct_text(entry, "lastName")
        title = direct_text(entry, "title")
        primary_name = " ".join(part for part in [first, last] if part).strip() or last or first or title
        entity_type = direct_text(entry, "sdnType")
        programs = "; ".join(sorted(set(all_desc_text(entry, "program"))))
        countries = "; ".join(sorted(set(all_desc_text(entry, "country"))))
        remarks = direct_text(entry, "remarks")
        names = []
        if primary_name:
            names.append((primary_name, "" , "primary"))
        for aka in entry.iter():
            if local_name(aka.tag) != "aka":
                continue
            aka_first = direct_text(aka, "firstName")
            aka_last = direct_text(aka, "lastName")
            aka_name = " ".join(part for part in [aka_first, aka_last] if part).strip() or aka_last or aka_first
            if aka_name:
                names.append((aka_name, aka_name, "alias"))
        for candidate_name, alias_value, name_type in names:
            records.append({
                "source_record_id": uid, "candidate_name": candidate_name,
                "normalized_candidate_name": normalize_name(candidate_name), "alias": alias_value,
                "name_type": name_type, "entity_type": entity_type, "program_or_list": programs,
                "country": countries, "remarks": remarks, "source": source_label, "source_type": "official_sanctions_list",
                "source_country": "United States", "source_url": source_url, "raw_file_path": str(path), "raw_file_sha256": raw_sha256,
            })
    elapsed = time.perf_counter() - start
    return pd.DataFrame(records).drop_duplicates(), elapsed


def parse_un_xml(path, source_label, source_url, raw_sha256):
    start = time.perf_counter()
    records = []
    root = ET.parse(path).getroot()
    for entry in root.iter():
        tag = local_name(entry.tag)
        if tag not in {"INDIVIDUAL", "ENTITY"}:
            continue
        reference = direct_text(entry, "REFERENCE_NUMBER") or direct_text(entry, "DATAID")
        parts = [direct_text(entry, field) for field in ["FIRST_NAME", "SECOND_NAME", "THIRD_NAME", "FOURTH_NAME"]]
        primary_name = " ".join(part for part in parts if part and part.lower() != "na").strip()
        entity_type = "Individual" if tag == "INDIVIDUAL" else "Entity"
        list_type = direct_text(entry, "UN_LIST_TYPE") or direct_text(entry, "LIST_TYPE")
        remarks = direct_text(entry, "COMMENTS1")
        countries = "; ".join(sorted(set(all_desc_text(entry, "NATIONALITY") + all_desc_text(entry, "COUNTRY"))))
        names = []
        if primary_name:
            names.append((primary_name, "", "primary"))
        for alias_tag in ["INDIVIDUAL_ALIAS", "ENTITY_ALIAS"]:
            for alias in entry.iter():
                if local_name(alias.tag) != alias_tag:
                    continue
                alias_name = direct_text(alias, "ALIAS_NAME")
                if alias_name and alias_name.lower() != "na":
                    names.append((alias_name, alias_name, "alias"))
        for candidate_name, alias_value, name_type in names:
            records.append({
                "source_record_id": reference, "candidate_name": candidate_name,
                "normalized_candidate_name": normalize_name(candidate_name), "alias": alias_value,
                "name_type": name_type, "entity_type": entity_type, "program_or_list": list_type,
                "country": countries, "remarks": remarks, "source": source_label, "source_type": "official_sanctions_list",
                "source_country": "United Nations", "source_url": source_url, "raw_file_path": str(path), "raw_file_sha256": raw_sha256,
            })
    elapsed = time.perf_counter() - start
    return pd.DataFrame(records).drop_duplicates(), elapsed


In [8]:
OFAC_CANDIDATES = [
    {"source": "OFAC SDN", "url": "https://www.treasury.gov/ofac/downloads/sdn.xml", "filename": "ofac_sdn.xml", "parser": parse_ofac_xml},
    {"source": "OFAC Consolidated", "url": "https://www.treasury.gov/ofac/downloads/consolidated/consolidated.xml", "filename": "ofac_consolidated.xml", "parser": parse_ofac_xml},
]
UN_CANDIDATES = [
    {"source": "UN Consolidated", "url": "https://scsanctions.un.org/resources/xml/en/consolidated.xml", "filename": "un_consolidated.xml", "parser": parse_un_xml},
]

benchmark_rows = []
parsed_frames = []
for candidate in OFAC_CANDIDATES + UN_CANDIDATES:
    raw_dir = OFAC_RAW_DIR if candidate["source"].startswith("OFAC") else UN_RAW_DIR
    result = fetch_url_once(candidate["url"], raw_dir / candidate["filename"], force=FORCE_DOWNLOAD)
    benchmark_rows.append({
        "notebook": "01_sanciones_oficiales_ofac_onu", "source": candidate["source"], "step": "download",
        "duration_seconds": round(result["seconds"], 6), "records_processed": 0, "records_per_second": 0,
        "status": result["status"], "error_message": result["error"], "notes": result["note"],
    })
    if result["ok"]:
        try:
            parsed_df, parse_seconds = candidate["parser"](result["path"], candidate["source"], candidate["url"], result["sha256"])
            parsed_frames.append(parsed_df)
            benchmark_rows.append({
                "notebook": "01_sanciones_oficiales_ofac_onu", "source": candidate["source"], "step": "parsing",
                "duration_seconds": round(parse_seconds, 6), "records_processed": len(parsed_df),
                "records_per_second": round(len(parsed_df) / parse_seconds, 2) if parse_seconds else 0,
                "status": "ok", "error_message": "", "notes": f"sha256={result['sha256']}",
            })
        except Exception as exc:
            benchmark_rows.append({
                "notebook": "01_sanciones_oficiales_ofac_onu", "source": candidate["source"], "step": "parsing",
                "duration_seconds": 0, "records_processed": 0, "records_per_second": 0,
                "status": "error", "error_message": str(exc)[:300], "notes": "parse failed",
            })

sanctions_df = pd.concat(parsed_frames, ignore_index=True) if parsed_frames else pd.DataFrame()
print(f"Registros sanciones/aliases parseados: {len(sanctions_df):,}")
display(sanctions_df.head(20) if not sanctions_df.empty else sanctions_df)


Registros sanciones/aliases parseados: 48,938


,source_record_id,candidate_name,normalized_candidate_name,alias,name_type,entity_type,program_or_list,country,remarks,source,source_type,source_country,source_url,raw_file_path,raw_file_sha256
0,36,AEROCARIBBEAN AIRLINES,aerocaribbean airlines,,primary,Entity,CUBA,Cuba,,OFAC SDN,official_sanctions_list,United States,https://www.treasury.gov/ofac/downloads/sdn.xml,C:\Users\Giovanny\Desktop\web_scraping_peps\no...,85030e0207480c7fe8d0abc3f75bbeab1fa217568b7e72...
1,36,AERO-CARIBBEAN,aero caribbean,AERO-CARIBBEAN,alias,Entity,CUBA,Cuba,,OFAC SDN,official_sanctions_list,United States,https://www.treasury.gov/ofac/downloads/sdn.xml,C:\Users\Giovanny\Desktop\web_scraping_peps\no...,85030e0207480c7fe8d0abc3f75bbeab1fa217568b7e72...
2,173,"ANGLO-CARIBBEAN CO., LTD.",anglo caribbean co ltd,,primary,Entity,CUBA,United Kingdom,,OFAC SDN,official_sanctions_list,United States,https://www.treasury.gov/ofac/downloads/sdn.xml,C:\Users\Giovanny\Desktop\web_scraping_peps\no...,85030e0207480c7fe8d0abc3f75bbeab1fa217568b7e72...
3,173,AVIA IMPORT,avia import,AVIA IMPORT,alias,Entity,CUBA,United Kingdom,,OFAC SDN,official_sanctions_list,United States,https://www.treasury.gov/ofac/downloads/sdn.xml,C:\Users\Giovanny\Desktop\web_scraping_peps\no...,85030e0207480c7fe8d0abc3f75bbeab1fa217568b7e72...
4,306,BANCO NACIONAL DE CUBA,banco nacional de cuba,,primary,Entity,CUBA,Japan; Panama; Spain; Switzerland,,OFAC SDN,official_sanctions_list,United States,https://www.treasury.gov/ofac/downloads/sdn.xml,C:\Users\Giovanny\Desktop\web_scraping_peps\no...,85030e0207480c7fe8d0abc3f75bbeab1fa217568b7e72...
5,306,BNC,bnc,BNC,alias,Entity,CUBA,Japan; Panama; Spain; Switzerland,,OFAC SDN,official_sanctions_list,United States,https://www.treasury.gov/ofac/downloads/sdn.xml,C:\Users\Giovanny\Desktop\web_scraping_peps\no...,85030e0207480c7fe8d0abc3f75bbeab1fa217568b7e72...
6,306,NATIONAL BANK OF CUBA,national bank of cuba,NATIONAL BANK OF CUBA,alias,Entity,CUBA,Japan; Panama; Spain; Switzerland,,OFAC SDN,official_sanctions_list,United States,https://www.treasury.gov/ofac/downloads/sdn.xml,C:\Users\Giovanny\Desktop\web_scraping_peps\no...,85030e0207480c7fe8d0abc3f75bbeab1fa217568b7e72...
7,424,BOUTIQUE LA MAISON,boutique la maison,,primary,Entity,CUBA,Panama,,OFAC SDN,official_sanctions_list,United States,https://www.treasury.gov/ofac/downloads/sdn.xml,C:\Users\Giovanny\Desktop\web_scraping_peps\no...,85030e0207480c7fe8d0abc3f75bbeab1fa217568b7e72...
8,475,CASA DE CUBA,casa de cuba,,primary,Entity,CUBA,Mexico; Spain,,OFAC SDN,official_sanctions_list,United States,https://www.treasury.gov/ofac/downloads/sdn.xml,C:\Users\Giovanny\Desktop\web_scraping_peps\no...,85030e0207480c7fe8d0abc3f75bbeab1fa217568b7e72...
9,480,"CECOEX, S.A.",cecoex s a,,primary,Entity,CUBA,Panama,,OFAC SDN,official_sanctions_list,United States,https://www.treasury.gov/ofac/downloads/sdn.xml,C:\Users\Giovanny\Desktop\web_scraping_peps\no...,85030e0207480c7fe8d0abc3f75bbeab1fa217568b7e72...


In [9]:
def evidence_row(seed, record, score):
    matched_fields, conflicting_fields = context_assessment(seed, record)
    confidence = identity_confidence(score, seed, matched_fields, sensitive=True)
    risk_level, recommended_action = risk_and_action(score, confidence, record.get("source_type", ""))
    source_name = record.get("source", "")
    signal_type = "un_sanctions_candidate" if source_name.startswith("UN") else "ofac_sanctions_candidate"
    source_country = record.get("source_country", "")
    evidence_summary = "; ".join(part for part in [
        f"record_id={record.get('source_record_id', '')}",
        f"name={record.get('candidate_name', '')}",
        f"type={record.get('entity_type', '')}",
        f"program/list={record.get('program_or_list', '')}",
        f"country={record.get('country', '')}",
    ] if part and not part.endswith("="))
    involvement_summary = "Coincidencia nominal en lista de sanciones; requiere revision por falta de identificadores adicionales."
    limitations = "Fuente sensible. No se confirma identidad sin identificadores adicionales como fecha de nacimiento, nacionalidad, RFC/CURP legalmente proporcionado u otro contexto fuerte."
    return {
        "seed_id": seed.get("seed_id", stable_hash(seed.get("nombre_persona_input", ""))),
        "nombre_persona_input": seed.get("nombre_persona_input", ""),
        "normalized_name_input": seed.get("normalized_name_input", normalize_name(seed.get("nombre_persona_input", ""))),
        "estado_input": seed.get("estado_input", ""), "municipio_input": seed.get("municipio_input", ""),
        "puesto_input": seed.get("puesto_input", ""), "dependencia_input": seed.get("dependencia_input", ""),
        "periodo_inicio_input": seed.get("periodo_inicio_input", ""), "periodo_fin_input": seed.get("periodo_fin_input", ""),
        "partido_input": seed.get("partido_input", ""), "rfc_input": seed.get("rfc_input", ""), "curp_input": seed.get("curp_input", ""),
        "empresa_relacionada_input": seed.get("empresa_relacionada_input", ""), "alias_input": seed.get("alias_input", ""),
        "source": source_name, "source_type": record.get("source_type", "official_sanctions_list"),
        "source_country": source_country, "source_official": True, "source_category": "sanctions", "source_reliability": "official_bulk_dataset",
        "matched_record_name": record.get("candidate_name", ""), "matched_record_normalized_name": record.get("normalized_candidate_name", ""),
        "matched_alias": record.get("alias", ""), "matched_entity_type": record.get("entity_type", ""), "matched_role": "", "matched_position": "",
        "matched_agency": record.get("program_or_list", ""), "matched_state": "", "matched_municipality": "", "matched_country": record.get("country", ""),
        "matched_identifier": record.get("source_record_id", ""), "matched_company": record.get("candidate_name", "") if str(record.get("entity_type", "")).lower() == "entity" else "",
        "matched_rfc": "", "matched_curp": "", "match_score_pct": score, "match_strength": classify_match(score),
        "match_method": match_method_for(seed.get("nombre_persona_input", ""), record.get("candidate_name", ""), alias=bool(record.get("alias", ""))),
        "matched_fields": matched_fields, "conflicting_fields": conflicting_fields,
        "requires_review": True, "identity_confidence_pct": confidence, "signal_type": signal_type,
        "signal_category": "official_sanctions_candidate", "severity": "official_sensitive_candidate",
        "risk_layer": "sanctions", "risk_level_candidate": risk_level, "confidence_pct": confidence,
        "evidence_title": f"{source_name} candidate: {record.get('candidate_name', '')}",
        "evidence_summary": evidence_summary, "evidence_snippet": str(record.get("remarks", ""))[:700],
        "evidence_keywords": "sanctions; official list", "evidence_date": "", "evidence_url": record.get("source_url", ""),
        "source_url": record.get("source_url", ""), "search_query": seed.get("nombre_persona_input", ""), "search_engine": "",
        "retrieved_at": now_utc(), "raw_file_path": record.get("raw_file_path", ""), "raw_file_sha256": record.get("raw_file_sha256", ""),
        "involvement_summary": involvement_summary, "explanation": "Candidato de coincidencia por similitud nominal contra lista oficial. No constituye confirmacion de identidad ni determinacion legal.",
        "limitations": limitations, "recommended_action": recommended_action,
    }


def match_sanctions(seeds, sanctions):
    if sanctions.empty:
        return empty_evidence_df(), 0.0, 0
    start = time.perf_counter()
    choices = sanctions["normalized_candidate_name"].fillna("").tolist()
    rows = []
    for _, seed in tqdm(seeds.iterrows(), total=len(seeds), desc="Matching sanciones"):
        seed_name = seed.get("normalized_name_input", normalize_name(seed.get("nombre_persona_input", "")))
        extracted = process.extract(seed_name, choices, scorer=fuzz.WRatio, score_cutoff=60, limit=30)
        for _, rough_score, idx in extracted:
            record = sanctions.iloc[idx]
            score = score_name_match(seed.get("nombre_persona_input", seed_name), record.get("candidate_name", ""))
            if classify_match(score) == "discard":
                continue
            rows.append(evidence_row(seed, record, score))
    elapsed = time.perf_counter() - start
    evidence = pd.DataFrame(rows, columns=EVIDENCE_COLUMNS) if rows else empty_evidence_df()
    if not evidence.empty:
        evidence = evidence.drop_duplicates(subset=["seed_id", "source", "matched_identifier", "matched_record_name", "matched_alias"], keep="first")
    return evidence, elapsed, len(seeds) * len(sanctions)

evidence_df, match_seconds, comparisons = match_sanctions(seeds_df, sanctions_df)
benchmark_rows.append({
    "notebook": "01_sanciones_oficiales_ofac_onu", "source": "OFAC/UN", "step": "matching",
    "duration_seconds": round(match_seconds, 6), "records_processed": int(comparisons),
    "records_per_second": round(comparisons / match_seconds, 2) if match_seconds else 0,
    "status": "ok", "error_message": "", "notes": f"evidence_rows={len(evidence_df)}",
})

evidence_path = OUTPUT_DIR / "01_sanciones_ofac_onu_evidence.csv"
evidence_df.to_csv(evidence_path, index=False, encoding="utf-8-sig")
print(f"Evidence guardado: {evidence_path}")
display(evidence_df.head(20))


Matching sanciones:   0%|          | 0/10 [00:00<?, ?it/s]

Matching sanciones:  10%|█         | 1/10 [00:00<00:01,  8.16it/s]

Matching sanciones:  20%|██        | 2/10 [00:00<00:00,  8.07it/s]

Matching sanciones:  30%|███       | 3/10 [00:00<00:00,  8.34it/s]

Matching sanciones:  40%|████      | 4/10 [00:00<00:00,  8.02it/s]

...

Matching sanciones:  90%|█████████ | 9/10 [00:01<00:00,  8.57it/s]

Matching sanciones: 100%|██████████| 10/10 [00:01<00:00,  8.45it/s]

Matching sanciones: 100%|██████████| 10/10 [00:01<00:00,  8.38it/s]

Evidence guardado: C:\Users\Giovanny\Desktop\web_scraping_peps\notebooks\output\01_sanciones_ofac_onu_evidence.csv


,seed_id,nombre_persona_input,normalized_name_input,estado_input,municipio_input,puesto_input,dependencia_input,periodo_inicio_input,periodo_fin_input,partido_input,...,source_url,search_query,search_engine,retrieved_at,raw_file_path,raw_file_sha256,involvement_summary,explanation,limitations,recommended_action
0,064afd0cc88b3699,Claudia Sheinbaum Pardo,claudia sheinbaum pardo,Nacional,,Presidenta de Mexico,Gobierno de Mexico,2024.0,,,...,https://www.treasury.gov/ofac/downloads/sdn.xml,Claudia Sheinbaum Pardo,,2026-05-07T07:52:30.787127+00:00,C:\Users\Giovanny\Desktop\web_scraping_peps\no...,85030e0207480c7fe8d0abc3f75bbeab1fa217568b7e72...,Coincidencia nominal en lista de sanciones; re...,Candidato de coincidencia por similitud nomina...,Fuente sensible. No se confirma identidad sin ...,discard_likely_false_positive
1,910a01d167e16711,Andres Manuel Lopez Obrador,andres manuel lopez obrador,Nacional,,Ex presidente de Mexico,Gobierno de Mexico,2018.0,2024.0,,...,https://www.treasury.gov/ofac/downloads/sdn.xml,Andres Manuel Lopez Obrador,,2026-05-07T07:52:30.910317+00:00,C:\Users\Giovanny\Desktop\web_scraping_peps\no...,85030e0207480c7fe8d0abc3f75bbeab1fa217568b7e72...,Coincidencia nominal en lista de sanciones; re...,Candidato de coincidencia por similitud nomina...,Fuente sensible. No se confirma identidad sin ...,discard_likely_false_positive
2,910a01d167e16711,Andres Manuel Lopez Obrador,andres manuel lopez obrador,Nacional,,Ex presidente de Mexico,Gobierno de Mexico,2018.0,2024.0,,...,https://www.treasury.gov/ofac/downloads/sdn.xml,Andres Manuel Lopez Obrador,,2026-05-07T07:52:30.910819+00:00,C:\Users\Giovanny\Desktop\web_scraping_peps\no...,85030e0207480c7fe8d0abc3f75bbeab1fa217568b7e72...,Coincidencia nominal en lista de sanciones; re...,Candidato de coincidencia por similitud nomina...,Fuente sensible. No se confirma identidad sin ...,discard_likely_false_positive
3,910a01d167e16711,Andres Manuel Lopez Obrador,andres manuel lopez obrador,Nacional,,Ex presidente de Mexico,Gobierno de Mexico,2018.0,2024.0,,...,https://www.treasury.gov/ofac/downloads/sdn.xml,Andres Manuel Lopez Obrador,,2026-05-07T07:52:30.911322+00:00,C:\Users\Giovanny\Desktop\web_scraping_peps\no...,85030e0207480c7fe8d0abc3f75bbeab1fa217568b7e72...,Coincidencia nominal en lista de sanciones; re...,Candidato de coincidencia por similitud nomina...,Fuente sensible. No se confirma identidad sin ...,discard_likely_false_positive
4,910a01d167e16711,Andres Manuel Lopez Obrador,andres manuel lopez obrador,Nacional,,Ex presidente de Mexico,Gobierno de Mexico,2018.0,2024.0,,...,https://www.treasury.gov/ofac/downloads/sdn.xml,Andres Manuel Lopez Obrador,,2026-05-07T07:52:30.911322+00:00,C:\Users\Giovanny\Desktop\web_scraping_peps\no...,85030e0207480c7fe8d0abc3f75bbeab1fa217568b7e72...,Coincidencia nominal en lista de sanciones; re...,Candidato de coincidencia por similitud nomina...,Fuente sensible. No se confirma identidad sin ...,discard_likely_false_positive
5,910a01d167e16711,Andres Manuel Lopez Obrador,andres manuel lopez obrador,Nacional,,Ex presidente de Mexico,Gobierno de Mexico,2018.0,2024.0,,...,https://www.treasury.gov/ofac/downloads/sdn.xml,Andres Manuel Lopez Obrador,,2026-05-07T07:52:30.913029+00:00,C:\Users\Giovanny\Desktop\web_scraping_peps\no...,85030e0207480c7fe8d0abc3f75bbeab1fa217568b7e72...,Coincidencia nominal en lista de sanciones; re...,Candidato de coincidencia por similitud nomina...,Fuente sensible. No se confirma identidad sin ...,discard_likely_false_positive
6,15b1d586cc02eb0e,Adan Augusto Lopez Hernandez,adan augusto lopez hernandez,Tabasco,,Senador / figura publica,Senado de la Republica,,,,...,https://www.treasury.gov/ofac/downloads/sdn.xml,Adan Augusto Lopez Hernandez,,2026-05-07T07:52:31.157629+00:00,C:\Users\Giovanny\Desktop\web_scraping_peps\no...,85030e0207480c7fe8d0abc3f75bbeab1fa217568b7e72...,Coincidencia nominal en lista de sanciones; re...,Candidato de coincidencia por similitud nomina...,Fuente sensible. No se confirma id

In [10]:
def top_row_for_person(subset):
    if subset.empty:
        return None
    temp = subset.copy()
    temp["_score"] = pd.to_numeric(temp["match_score_pct"], errors="coerce").fillna(0)
    temp["_confidence"] = pd.to_numeric(temp["identity_confidence_pct"], errors="coerce").fillna(0)
    return temp.sort_values(["_score", "_confidence"], ascending=False).iloc[0]

summary_rows = []
for _, seed in seeds_df.iterrows():
    subset = evidence_df[evidence_df["seed_id"].astype(str) == str(seed.get("seed_id", ""))] if not evidence_df.empty else empty_evidence_df()
    top = top_row_for_person(subset)
    total = len(subset)
    max_match = pd.to_numeric(subset["match_score_pct"], errors="coerce").fillna(0).max() if total else 0
    max_conf = pd.to_numeric(subset["identity_confidence_pct"], errors="coerce").fillna(0).max() if total else 0
    ofac_count = int(subset["source"].astype(str).str.contains("OFAC", case=False, na=False).sum()) if total else 0
    un_count = int(subset["source"].astype(str).str.contains("UN", case=False, na=False).sum()) if total else 0
    if total == 0:
        action = "no_action"
        review = bool(input_has_only_name(seed))
    elif max_match < 75:
        action = "discard_likely_false_positive"
        review = True
    elif max_match < 88:
        action = "manual_review"
        review = True
    else:
        action = "verify_identity"
        review = True
    summary_rows.append({
        "seed_id": seed.get("seed_id", stable_hash(seed.get("nombre_persona_input", ""))),
        "nombre_persona_input": seed.get("nombre_persona_input", ""),
        "total_sanctions_candidates": total,
        "ofac_candidates": ofac_count,
        "un_candidates": un_count,
        "max_match_score_pct": round(float(max_match), 2),
        "max_identity_confidence_pct": round(float(max_conf), 2),
        "max_severity": top.get("severity", "") if top is not None else "",
        "requires_review": review,
        "top_evidence_url": top.get("evidence_url", "") if top is not None else "",
        "top_involvement_summary": top.get("involvement_summary", "") if top is not None else "",
        "recommended_action": action,
    })
summary_df = pd.DataFrame(summary_rows, columns=SUMMARY_COLUMNS)
summary_path = OUTPUT_DIR / "01_sanciones_ofac_onu_person_summary.csv"
benchmark_path = OUTPUT_DIR / "01_benchmark_sanciones.csv"
summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")
pd.DataFrame(benchmark_rows).to_csv(benchmark_path, index=False, encoding="utf-8-sig")
print(f"Summary guardado: {summary_path}")
print(f"Benchmark guardado: {benchmark_path}")
display(summary_df)
display(pd.DataFrame(benchmark_rows))


Summary guardado: C:\Users\Giovanny\Desktop\web_scraping_peps\notebooks\output\01_sanciones_ofac_onu_person_summary.csv
Benchmark guardado: C:\Users\Giovanny\Desktop\web_scraping_peps\notebooks\output\01_benchmark_sanciones.csv


,seed_id,nombre_persona_input,total_sanctions_candidates,ofac_candidates,un_candidates,max_match_score_pct,max_identity_confidence_pct,max_severity,requires_review,top_evidence_url,top_involvement_summary,recommended_action
0,064afd0cc88b3699,Claudia Sheinbaum Pardo,1,1,0,74.0,41.8,official_sensitive_candidate,True,https://www.treasury.gov/ofac/downloads/sdn.xml,Coincidencia nominal en lista de sanciones; re...,discard_likely_false_positive
1,910a01d167e16711,Andres Manuel Lopez Obrador,5,5,0,74.0,41.8,official_sensitive_candidate,True,https://www.treasury.gov/ofac/downloads/sdn.xml,Coincidencia nominal en lista de sanciones; re...,discard_likely_false_positive
2,55c8819cc4622591,Marcelo Ebrard Casaubon,0,0,0,0.0,0.0,,False,,,no_action
3,15b1d586cc02eb0e,Adan Augusto Lopez Hernandez,6,6,0,74.0,41.8,official_sensitive_candidate,True,https://www.treasury.gov/ofac/downloads/sdn.xml,Coincidencia nominal en lista de sanciones; re...,discard_likely_false_positive
4,2597a81eb6062feb,Ricardo Monreal Avila,0,0,0,0.0,0.0,,False,,,no_action
5,39162acdaa914138,Xochitl Galvez Ruiz,0,0,0,0.0,0.0,,False,,,no_action
6,0cafd5ecac30f555,Samuel Garcia Sepulveda,1,1,0,74.0,41.8,official_sensitive_candidate,True,https://www.treasury.gov/ofac/downloads/sdn.xml,Coincidencia nominal en lista de sanciones; re...,discard_likely_false_positive
7,cba9b1a8ead15ac3,Clara Brugada Molina,0,0,0,0.0,0.0,,False,,,no_action
8,69573f0325299b8b,Omar Garcia Harfuch,17,17,0,74.0,41.8,official_sensitive_candidate,True,https://www.treasury.gov/ofac/downloads/sdn.xml,Coincidencia nominal en lista de sanciones; re...,discard_likely_false_positive
9,75c154f2e7b3a23f,Luisa Maria Alcalde Lujan,8,8,0,74.0,41.8,official_sensitive_candidate,True,https://www.treasury.gov/ofac/downloads/sdn.xml,Coincidencia nominal en lista de sanciones; re...,discard_likely_false_positive


,notebook,source,step,duration_seconds,records_processed,records_per_second,status,error_message,notes
0,01_sanciones_oficiales_ofac_onu,OFAC SDN,download,0.000000,0,0.00,cached,,raw reutilizado
1,01_sanciones_oficiales_ofac_onu,OFAC SDN,parsing,3.502467,43581,12442.94,ok,,sha256=85030e0207480c7fe8d0abc3f75bbeab1fa2175...
2,01_sanciones_oficiales_ofac_onu,OFAC Consolidated,download,0.000000,0,0.00,cached,,raw reutilizado
3,01_sanciones_oficiales_ofac_onu,OFAC Consolidated,parsing,0.087908,1586,18041.61,ok,,sha256=466179521ec7a28f6d588584787a0144bc79e86...
4,01_sanciones_oficiales_ofac_onu,UN Consolidated,download,0.000000,0,0.00,cached,,raw reutilizado
5,01_sanciones_oficiales_ofac_onu,UN Consolidated,parsing,0.165399,3771,22799.47,ok,,sha256=864e6256c9e17f77af600a17a12872af5498208...
6,01_sanciones_oficiales_ofac_onu,OFAC/UN,matching,1.206963,489380,405463.79,ok,,evidence_rows=38


In [11]:
print("Resumen notebook 01")
print(f"1. Personas procesadas: {len(seeds_df):,}")
print(f"2. Filas de evidencia generadas: {len(evidence_df):,}")
print(f"3. Personas con hits: {evidence_df['seed_id'].nunique() if not evidence_df.empty else 0:,}")
print("4. Top 10 filas tabulares:")
display(evidence_df.head(10))
print("5. CSV generados:")
for path in [evidence_path, summary_path, benchmark_path]:
    print(f"- {path}")


Resumen notebook 01
1. Personas procesadas: 10
2. Filas de evidencia generadas: 38
3. Personas con hits: 6
4. Top 10 filas tabulares:


,seed_id,nombre_persona_input,normalized_name_input,estado_input,municipio_input,puesto_input,dependencia_input,periodo_inicio_input,periodo_fin_input,partido_input,...,source_url,search_query,search_engine,retrieved_at,raw_file_path,raw_file_sha256,involvement_summary,explanation,limitations,recommended_action
0,064afd0cc88b3699,Claudia Sheinbaum Pardo,claudia sheinbaum pardo,Nacional,,Presidenta de Mexico,Gobierno de Mexico,2024.0,,,...,https://www.treasury.gov/ofac/downloads/sdn.xml,Claudia Sheinbaum Pardo,,2026-05-07T07:52:30.787127+00:00,C:\Users\Giovanny\Desktop\web_scraping_peps\no...,85030e0207480c7fe8d0abc3f75bbeab1fa217568b7e72...,Coincidencia nominal en lista de sanciones; re...,Candidato de coincidencia por similitud nomina...,Fuente sensible. No se confirma identidad sin ...,discard_likely_false_positive
1,910a01d167e16711,Andres Manuel Lopez Obrador,andres manuel lopez obrador,Nacional,,Ex presidente de Mexico,Gobierno de Mexico,2018.0,2024.0,,...,https://www.treasury.gov/ofac/downloads/sdn.xml,Andres Manuel Lopez Obrador,,2026-05-07T07:52:30.910317+00:00,C:\Users\Giovanny\Desktop\web_scraping_peps\no...,85030e0207480c7fe8d0abc3f75bbeab1fa217568b7e72...,Coincidencia nominal en lista de sanciones; re...,Candidato de coincidencia por similitud nomina...,Fuente sensible. No se confirma identidad sin ...,discard_likely_false_positive
2,910a01d167e16711,Andres Manuel Lopez Obrador,andres manuel lopez obrador,Nacional,,Ex presidente de Mexico,Gobierno de Mexico,2018.0,2024.0,,...,https://www.treasury.gov/ofac/downloads/sdn.xml,Andres Manuel Lopez Obrador,,2026-05-07T07:52:30.910819+00:00,C:\Users\Giovanny\Desktop\web_scraping_peps\no...,85030e0207480c7fe8d0abc3f75bbeab1fa217568b7e72...,Coincidencia nominal en lista de sanciones; re...,Candidato de coincidencia por similitud nomina...,Fuente sensible. No se confirma identidad sin ...,discard_likely_false_positive
3,910a01d167e16711,Andres Manuel Lopez Obrador,andres manuel lopez obrador,Nacional,,Ex presidente de Mexico,Gobierno de Mexico,2018.0,2024.0,,...,https://www.treasury.gov/ofac/downloads/sdn.xml,Andres Manuel Lopez Obrador,,2026-05-07T07:52:30.911322+00:00,C:\Users\Giovanny\Desktop\web_scraping_peps\no...,85030e0207480c7fe8d0abc3f75bbeab1fa217568b7e72...,Coincidencia nominal en lista de sanciones; re...,Candidato de coincidencia por similitud nomina...,Fuente sensible. No se confirma identidad sin ...,discard_likely_false_positive
4,910a01d167e16711,Andres Manuel Lopez Obrador,andres manuel lopez obrador,Nacional,,Ex presidente de Mexico,Gobierno de Mexico,2018.0,2024.0,,...,https://www.treasury.gov/ofac/downloads/sdn.xml,Andres Manuel Lopez Obrador,,2026-05-07T07:52:30.911322+00:00,C:\Users\Giovanny\Desktop\web_scraping_peps\no...,85030e0207480c7fe8d0abc3f75bbeab1fa217568b7e72...,Coincidencia nominal en lista de sanciones; re...,Candidato de coincidencia por similitud nomina...,Fuente sensible. No se confirma identidad sin ...,discard_likely_false_positive
5,910a01d167e16711,Andres Manuel Lopez Obrador,andres manuel lopez obrador,Nacional,,Ex presidente de Mexico,Gobierno de Mexico,2018.0,2024.0,,...,https://www.treasury.gov/ofac/downloads/sdn.xml,Andres Manuel Lopez Obrador,,2026-05-07T07:52:30.913029+00:00,C:\Users\Giovanny\Desktop\web_scraping_peps\no...,85030e0207480c7fe8d0abc3f75bbeab1fa217568b7e72...,Coincidencia nominal en lista de sanciones; re...,Candidato de coincidencia por similitud nomina...,Fuente sensible. No se confirma identidad sin ...,discard_likely_false_positive
6,15b1d586cc02eb0e,Adan Augusto Lopez Hernandez,adan augusto lopez hernandez,Tabasco,,Senador / figura publica,Senado de la Republica,,,,...,https://www.treasury.gov/ofac/downloads/sdn.xml,Adan Augusto Lopez Hernandez,,2026-05-07T07:52:31.157629+00:00,C:\Users\Giovanny\Desktop\web_scraping_peps\no...,85030e0207480c7fe8d0abc3f75bbeab1fa217568b7e72...,Coincidencia nominal en lista de sanciones; re...,Candidato de coincidencia por similitud nomina...,Fuente sensible. No se confirma id

5. CSV generados:
- C:\Users\Giovanny\Desktop\web_scraping_peps\notebooks\output\01_sanciones_ofac_onu_evidence.csv
- C:\Users\Giovanny\Desktop\web_scraping_peps\notebooks\output\01_sanciones_ofac_onu_person_summary.csv
- C:\Users\Giovanny\Desktop\web_scraping_peps\notebooks\output\01_benchmark_sanciones.csv
